<a href="https://colab.research.google.com/github/Niladri-Baksi/SpeakEdge/blob/main/SpeakEdge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modules
1. Setup & Install
2. Audio Input (upload)
3. Speech-to-Text (Whisper)
4. Feature Extraction
5. Scoring Algorithm
6. Feedback + Recommendations

### Module 1

In [ ]:
!pip install openai-whisper
!pip install transformers
!pip install sentence-transformers
!pip install librosa
!pip install soundfile

!pip install --upgrade transformers
from transformers import pipeline

In [ ]:
import whisper
import librosa
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

### Module 2

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(audio_file)

### Module 3

In [ ]:
model = whisper.load_model("base")

In [ ]:
#STT
result = model.transcribe(audio_file)
transcript = result["text"]

print("Transcript:")
print(transcript)

### Module 4

In [ ]:
#Basic text features

import re

words = transcript.split()
total_words = len(words)
unique_words = len(set(words))

print("Total words:", total_words)
print("Unique words:", unique_words)

In [ ]:
#Filler word detection
filler_words = ["um", "uh", "like", "you know", "basically"]

filler_count = sum(transcript.lower().count(f) for f in filler_words)

print("Filler count:", filler_count)

In [ ]:
#Speech duration + WPM
audio, sr = librosa.load(audio_file)
duration = librosa.get_duration(y=audio, sr=sr)

wpm = (total_words / duration) * 60

print("Duration:", duration)
print("Words per minute:", wpm)

In [ ]:
# #Grammar checking (basic)
# grammar_pipeline = pipeline("text2text-generation", model="vennify/t5-base-grammar-correction")

# corrected = grammar_pipeline(transcript, max_length=512)[0]['generated_text']

# print("Corrected Text:")
# print(corrected)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("vennify/t5-base-grammar-correction")
model = AutoModelForSeq2SeqLM.from_pretrained("vennify/t5-base-grammar-correction")

input_text = "fix grammar: " + transcript

inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_length=512)

corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(corrected)

In [ ]:
#Coherence
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

question = "Tell me about yourself"

emb1 = model_embed.encode(question, convert_to_tensor=True)
emb2 = model_embed.encode(transcript, convert_to_tensor=True)

similarity = util.cos_sim(emb1, emb2).item()

print("Similarity score:", similarity)